
# quantization_final v2 — универсальная INT8-квантизация на Kaggle CPU

**Вход (любой из трёх, `MODEL_INPUT` в ячейке 2):**
- `checkpoint` — nnUNetv2-чекпоинт **любой архитектуры** (дистиллят, прунинг, другой тренер): сеть строится из plans; `ARCH_OVERRIDE` — для резанных ширин, которых plans не знает;
- `onnx` — готовый ONNX-файл;
- `ir` — готовый OpenVINO IR (.xml + .bin).

Обязательно приаттачен источник b2nd-препроцессинга (калибровка) — например kernel_source `alenasnigireva/nnunet` — и источник чекпоинта (ядро/датасет с .pth + plans.json).

**Выход:** OpenVINO INT8 IR (`quantized_model.xml/.bin`, MIXED = W8A8) + `model_meta.json` + `selection_report.json`.

**Калибровка (Калибровка 2.0 — Табл. 4 отчёта v4):** 256 **показательных** патчей, не воздуха:
1. сетка кандидатов с шагом patch/2 по всем b2nd-томам;
2. отсечка воздуха: std патча и доля вокселей выше адаптивного тканевого порога (p10 + 0.30·(p90−p10) объёма);
3. квантильная стратификация по насыщенности + round-robin между кейсами — партия покрывает и богатые тканью, и периферию с частью контента (урок iterative-кризиса: fg-only калибровка не покрывает прод-диапазоны активаций);
4. 8 held-out патчей (не в NNCF) идут в argmax-гейт INT8 vs FP32.

Домен: b2nd уже CTNormalized — **никаких нормализаций и транспонов**; b2nd хранит **(1, Z, Y, X)**, Z — короткая ось. 48→256 патчей = стабильно +0.10 п.п. Dice (Табл. 2 отчёта v4). NNCF ~12 с/патч на 4 vCPU → 256 патчей ≈ 50 мин, в лимит сессии 12 ч укладывается.

**Чекпойнтование/продолжение:** каждый этап кладёт маркер `.stage_*_done` в /kaggle/working; повторный запуск в той же сессии пропускает сделанное (calib → export → quant). **Остановка при нештатном поведении:** страница ядра → ⋮ → **Stop** (не ждать ошибки). RAM печатается на каждом этапе.

**Стек (`STACK`, ячейка 1):** `'prod'` = nncf 2.19 + OV 2025.2 (прод-пины, валидировано) / `'v3x'` = nncf 3.4 + OV 2026.4 (гейт идентичен, выигрыша нет — BENCHMARKS.md).


In [ ]:

# ---------- Ячейка 1: окружение (пип-установки + хелперы + RAM-монитор) ----------
import functools
import glob
import json
import os
import shutil
import subprocess
import sys
import time

print = functools.partial(print, flush=True)

def avail_gb():
    with open('/proc/meminfo') as f:
        for line in f:
            if line.startswith('MemAvailable'):
                return int(line.split()[1]) / 2**20
    return -1

def ram():
    return f'{avail_gb():.1f} ГБ свободно'

def find_one(pattern):
    hits = sorted(glob.glob(pattern, recursive=True))
    return hits[0] if hits else None

def find_any(*patterns):
    for p in patterns:
        h = find_one(p)
        if h:
            return h
    return None

os.environ.setdefault('TF_CPP_MIN_LOG_LEVEL', '3')
sys.modules.setdefault('tensorflow', None)   # NNCF иначе пытается импортировать TF
sys.modules.setdefault('tensorboard', None)

def _pip(*args):
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', *args], check=True)

STACK = 'prod'  # 'prod' = nncf 2.19 + ov 2025.2 (прод-пины, валидировано)
# 'v3x' = nncf 3.4 + ov 2026.4 — гейт идентичен, выигрыша нет (BENCHMARKS.md)
WANT_NNCF = '3.4.0' if STACK == 'v3x' else '2.19.0'
WANT_OV = '2026.4.1' if STACK == 'v3x' else '2025.2.0'
print(f'[stack] {STACK}: цель nncf=={WANT_NNCF}, openvino=={WANT_OV}')

try:
    import nncf
except ImportError:
    print(f'[stack] nncf отсутствует — ставлю {WANT_NNCF} ...')
    _pip(f'nncf=={WANT_NNCF}')
    import nncf
if nncf.__version__ != WANT_NNCF:
    print(f'[stack] nncf {nncf.__version__} -> {WANT_NNCF} ...')
    _pip(f'nncf=={WANT_NNCF}')
    import importlib as _iln
    _iln.reload(nncf)
print('[stack] nncf', nncf.__version__)

try:
    import openvino as ov
except ImportError:
    print(f'[stack] openvino отсутствует — ставлю {WANT_OV} ...')
    _pip(f'openvino=={WANT_OV}')
    import openvino as ov
if ov.__version__ != WANT_OV:
    print(f'[stack] openvino {ov.__version__} -> {WANT_OV} ...')
    _pip(f'openvino=={WANT_OV}')
    import importlib as _ilo
    _ilo.reload(ov)
print('[stack] openvino', ov.__version__)

try:
    import nnunetv2
    if not hasattr(nnunetv2, '__version__'):
        nnunetv2.__version__ = '2.6.2'
except ImportError:
    # нужны только PlansManager/build_network_architecture — есть во всех nnUNetv2 2.x
    _pip('nnunetv2==2.6.2')

# onnxscript нужен dynamo-экспортёру torch.onnx в torch>=2.9 (на Kaggle-образе нет)
try:
    import onnxscript
except ImportError:
    print('ставлю onnxscript (нужен ONNX-экспортёру torch 2.11) ...')
    _pip('onnxscript')

import numpy as np
import torch
import openvino as ov
import nncf
print('openvino', ov.__version__, '| torch', torch.__version__, '| nncf', nncf.__version__)
print('RAM:', ram())

WORK = '/kaggle/working'

def stage_done(name):
    """Файл-маркер завершённого этапа: при повторном запуске этап пропускается."""
    return os.path.isfile(os.path.join(WORK, f'.stage_{name}_done'))

def mark_stage(name):
    open(os.path.join(WORK, f'.stage_{name}_done'), 'w').write('ok')


In [ ]:

# ---------- Ячейка 2: КОНФИГУРАЦИЯ (единственная ячейка для правок) ----------
# Универсальный вход: любой nnUNetv2-чекпоинт (дистиллят/прунинг/другая
# архитектура — сеть строится из plans) или готовый ONNX / OpenVINO IR.
MODEL_INPUT = 'checkpoint'   # 'checkpoint' | 'onnx' | 'ir'

CKPT_PATH = ''               # прямой путь к .pth; пусто = автопоиск по /kaggle/input
CKPT_HINT = 'checkpoint_latest.pth'  # дистиллят 55ep из датасета distill55ep
MODEL_FILE = ''              # готовый .onnx или .xml (+ .bin рядом) при MODEL_INPUT != 'checkpoint'

CONFIG_NAME = '3d_fullres'   # имя конфигурации в plans
ARCH_OVERRIDE = None         # резанные ширины, если plans их не знает, например:
                             # {'features_per_stage': [32,58,115,256,256,256],
                             #  'n_conv_per_stage_encoder': [...], 'n_conv_per_stage_decoder': [...]}

# --- калибровка: показательные патчи (Калибровка 2.0, Табл. 4 отчёта v4) ---
K_CALIB = 256                # дефолт v4: 48 -> 256 = +0.10 п.п. Dice стабильно
SCAN_STRIDE_DIV = 2          # сетка кандидатов с шагом patch//2 по каждой оси
MIN_STD = 0.2                # std патча ниже => «плоский» (воздух/однородный) — отброс
MIN_TISSUE_FRAC = 0.02       # <2% вокселей выше тканевого порога => воздух — отброс
TISSUE_FRAC_LO = 0.30        # тканевый порог = p10 + 0.30*(p90-p10) объёма (адаптивно)
N_GATE = 8                   # held-out патчи для argmax-гейта (не участвуют в NNCF)

STAMP = time.strftime('%Y%m%d_%H%M%S')
print('MODEL_INPUT =', MODEL_INPUT, '| K_CALIB =', K_CALIB, '| штамп:', STAMP)
assert MODEL_INPUT in ('checkpoint', 'onnx', 'ir')


In [ ]:

# ---------- Ячейка 3: вход модели (чекпоинт+plans / ONNX / IR) ----------
state, plans, features = None, None, None
CKPT, trainer_name, epoch = None, '?', '?'

if MODEL_INPUT == 'checkpoint':
    all_pth = sorted(glob.glob('/kaggle/input/**/*.pth', recursive=True))
    print('найдено .pth:', len(all_pth))
    for p in all_pth:
        print('  ', p)
    if CKPT_PATH:
        assert os.path.isfile(CKPT_PATH), f'CKPT_PATH не найден: {CKPT_PATH}'
        cands = [CKPT_PATH]
    else:
        cands = [p for p in all_pth if CKPT_HINT in p] if CKPT_HINT else all_pth
    assert len(cands) == 1, ('неоднозначный выбор чекпоинта: ' + repr(cands) +
                             ' — задайте CKPT_HINT (подстрока пути) или CKPT_PATH. '
                             'Осторожно: в маунтах бывают учителя (эпоха 1000) — проверьте эпоху ниже.')
    CKPT = cands[0]
    ck = torch.load(CKPT, map_location='cpu', weights_only=False)
    epoch = ck.get('current_epoch', '?')
    trainer_name = str(ck.get('trainer_name', '?'))
    print('чекпоинт:', CKPT)
    print('тренер:', trainer_name, '| эпоха:', epoch)

    state = None
    for k in ('network_weights', 'state_dict', 'model_state_dict', 'model'):
        v = ck.get(k)
        if isinstance(v, dict) and len(v):
            state = v
            break
    assert state is not None, 'в чекпоинте нет весов (network_weights/state_dict/...)'
    state = {k[10:] if k.startswith('_orig_mod.') else k: v for k, v in state.items()}

    # plans: из чекпоинта (новые nnUNet) или plans.json рядом / в маунтах
    if isinstance(ck.get('plans'), dict):
        plans = ck['plans']
        print('plans: взяты из чекпоинта')
    else:
        near = (glob.glob(os.path.join(os.path.dirname(CKPT), 'plans.json')) or
                glob.glob(os.path.join(os.path.dirname(os.path.dirname(CKPT)), 'plans.json')) or
                sorted(glob.glob('/kaggle/input/**/plans.json', recursive=True)))
        assert near, 'plans.json не найден — приаттачьте источник с plans.json'
        plans = json.load(open(near[0]))
        print('plans:', near[0])

    cfgs = plans['configurations']
    if CONFIG_NAME not in cfgs:
        CONFIG_NAME = sorted(cfgs)[0]
        print('!! CONFIG_NAME нет в plans — беру', CONFIG_NAME, '| доступно:', sorted(cfgs))
    cfg = cfgs[CONFIG_NAME]
    PATCH = tuple(int(v) for v in cfg['patch_size'])
    assert len(PATCH) == 3, f'только 3D-модели, patch_size={PATCH}'
    features = cfg['architecture']['arch_kwargs'].get('features_per_stage')
    print('patch_size:', PATCH, '| features_per_stage:', features)
else:
    assert MODEL_FILE and os.path.isfile(MODEL_FILE), f'MODEL_FILE не найден: {MODEL_FILE}'
    _m = ov.Core().read_model(MODEL_FILE)
    try:
        _sh = [d.get_length() if hasattr(d, 'get_length') else int(d)
               for d in _m.input(0).shape]
    except Exception as e:
        raise AssertionError(f'не удалось прочитать форму входа {MODEL_FILE}: {e}')
    assert all(v > 0 for v in _sh) and len(_sh) == 5 and _sh[1] == 1, \
        f'ожидается статический вход (1,1,Z,Y,X): {_sh}'
    PATCH = tuple(int(v) for v in _sh[2:])
    assert len(PATCH) == 3, f'только 3D-модели, patch_size={PATCH}'
    trainer_name = os.path.splitext(os.path.basename(MODEL_FILE))[0]
    print('вход модели:', _sh, '| PATCH =', PATCH)

print('RAM:', ram())


In [ ]:

# ---------- Ячейка 4: диагностика монтирований (видна всегда) ----------
print('дерево /kaggle/input (4 уровня, без b2nd/pkl/архивов):')
for root, dirs, files in os.walk('/kaggle/input'):
    depth = root[len('/kaggle/input'):].count(os.sep)
    if depth >= 4:
        dirs[:] = []
    show = [f for f in files if not f.endswith(('.b2nd', '.pkl'))][:6]
    if show or depth <= 1:
        print(' ', root, '->', show)


In [ ]:

# ---------- Ячейка 5: калибровка — показательные патчи (не воздуха) ----------
# Калибровка 2.0 (Табл. 4 отчёта v4): сетка кандидатов по b2nd-томам, отсечка
# воздуха (std + доля ткани выше адаптивного порога), квантильная стратификация
# по насыщенности + round-robin между кейсами. Партия покрывает и богатые
# тканью патчи, и периферию с частью контента (урок iterative-кризиса:
# fg-only калибровка не покрывает прод-диапазоны активаций).
# Патчи НЕ нормализуются: b2nd уже CTNormalized (домен == прод-инференсу).
# Гарантия худших классов через _seg.b2nd — следующий шаг при необходимости.
import blosc2

blosc2.set_nthreads(1)
CACHE = os.path.join(WORK, 'calibration_patches.npz')

if stage_done('calib') and os.path.isfile(CACHE):
    _z = np.load(CACHE)
    X_calib, X_gate = _z['patches'], _z['gate']
    print(f'кэш калибровки: {X_calib.shape}, гейт-патчи: {X_gate.shape}')
else:
    DATA_DIR = find_one('/kaggle/input/**/nnUNet_preprocessed/*/*_3d_fullres')
    assert DATA_DIR, 'не найден препроцессинг b2nd (nnUNet_preprocessed/<Dataset>/*_3d_fullres)'
    cases = [f for f in sorted(glob.glob(os.path.join(DATA_DIR, '*.b2nd')))
             if not f.endswith('_seg.b2nd')]
    print('папка b2nd:', DATA_DIR)
    print('случаев:', len(cases))
    assert cases, 'b2nd-томов нет — проверить маунт'

    pz, py, px = PATCH
    sz, sy, sx = (max(1, d // SCAN_STRIDE_DIV) for d in PATCH)
    cand, t0 = [], time.time()
    for ci, fp in enumerate(cases):
        arr = np.asarray(blosc2.open(fp))
        if arr.ndim == 4 and arr.shape[0] == 1:
            arr = arr[0]                      # (1, Z, Y, X) -> (Z, Y, X), без транспонов
        assert arr.ndim == 3 and arr.shape[0] < arr.shape[1], f'оси b2nd не (Z,Y,X): {arr.shape}'
        assert abs(float(arr.mean())) < 20, f'не CTNormalized-домен: mean={arr.mean():.2f}'
        lo, hi = np.percentile(arr, [10, 90])
        th = float(lo) + TISSUE_FRAC_LO * (float(hi) - float(lo))
        nz, ny, nx = arr.shape
        for z in range(0, max(1, nz - pz + 1), sz):
            for y in range(0, max(1, ny - py + 1), sy):
                for x in range(0, max(1, nx - px + 1), sx):
                    p = arr[z:z + pz, y:y + py, x:x + px]
                    if p.shape != (pz, py, px):
                        continue
                    cand.append((ci, z, y, x, float(p.std()), float((p > th).mean())))
        if (ci + 1) % 5 == 0 or ci == len(cases) - 1:
            print(f'[{ci + 1}/{len(cases)}] {os.path.basename(fp)}: {arr.shape} '
                  f'(порог ткани {th:.2f}) -> кандидатов {len(cand)}, '
                  f'{time.time() - t0:.0f} с, {ram()}')
        del arr

    good = [c for c in cand if c[4] >= MIN_STD and c[5] >= MIN_TISSUE_FRAC]
    print(f'кандидатов: {len(cand)}, после отсечки воздуха: {len(good)}')
    hist, edges = np.histogram([c[5] for c in good], bins=10)
    for h, e in zip(hist, edges):
        print(f'  frac>{e:.2f}: {int(h)}')
    assert good, 'все кандидаты отфильтрованы как воздух — ослабьте MIN_STD/MIN_TISSUE_FRAC'

    by_case = {}
    for c in good:
        by_case.setdefault(c[0], []).append(c)
    per = max(1, K_CALIB // len(by_case))
    pools, sel = {}, []
    for ci in sorted(by_case):
        lst = sorted(by_case[ci], key=lambda c: (c[5], c[1]))  # насыщенность, затем Z
        idx = np.unique(np.round(np.linspace(0, len(lst) - 1, min(per, len(lst)))).astype(int))
        pools[ci] = [lst[i] for i in idx]
    while len(sel) < K_CALIB and any(pools.values()):          # round-robin между кейсами
        for ci in sorted(pools):
            if pools[ci] and len(sel) < K_CALIB:
                sel.append(pools[ci].pop(0))
    taken = {(c[0], c[1], c[2], c[3]) for c in sel}
    rest = [c for c in good if (c[0], c[1], c[2], c[3]) not in taken]
    if len(sel) < K_CALIB:
        add = sorted(rest, key=lambda c: -c[5])[:K_CALIB - len(sel)]
        sel += add
        taken |= {(c[0], c[1], c[2], c[3]) for c in add}
        print(f'докинуто из остатка: {len(add)}')
    gsel = [c for c in sorted(rest, key=lambda c: -c[5])
            if (c[0], c[1], c[2], c[3]) not in taken][:N_GATE]
    per_case_counts = {}
    for c in sel:
        per_case_counts[c[0]] = per_case_counts.get(c[0], 0) + 1
    print(f'выбрано {len(sel)} калибровочных + {len(gsel)} гейт-патчей; по кейсам: {per_case_counts}')
    print(f'frac выбранной партии: {min(c[5] for c in sel):.2f}..{max(c[5] for c in sel):.2f}')

    groups = {}
    for tag, lst in (('c', sel), ('g', gsel)):
        for c in lst:
            groups.setdefault(c[0], []).append((tag, c))
    X_c, X_g = [], []
    for ci, items in sorted(groups.items()):
        arr = np.asarray(blosc2.open(cases[ci]))
        if arr.ndim == 4 and arr.shape[0] == 1:
            arr = arr[0]
        for tag, c in items:
            p = arr[c[1]:c[1] + pz, c[2]:c[2] + py, c[3]:c[3] + px].astype(np.float32)
            (X_c if tag == 'c' else X_g).append(p)
        del arr
        print(f'материализация кейса {ci}: {len(items)} патчей, {ram()}')

    X_calib = np.stack(X_c)[:, np.newaxis]
    X_gate = (np.stack(X_g)[:, np.newaxis] if X_g
              else np.zeros((0, 1, pz, py, px), np.float32))
    assert X_calib.shape[1:] == (1, pz, py, px), \
        f'форма патча сломана (ось канала дважды?): {X_calib.shape[1:]}'
    assert abs(float(X_calib.mean())) < 20, 'данные не похожи на CTNormalization-домен'
    np.savez(CACHE, patches=X_calib, gate=X_gate)
    json.dump({'k_requested': K_CALIB, 'k_used': len(sel), 'gate': len(gsel),
               'candidates': len(cand), 'non_air': len(good), 'cases': len(cases),
               'min_std': MIN_STD, 'min_tissue_frac': MIN_TISSUE_FRAC,
               'tissue_frac_lo': TISSUE_FRAC_LO, 'per_case_counts': per_case_counts},
              open(os.path.join(WORK, 'selection_report.json'), 'w'), indent=1)
    mark_stage('calib')
    print('кэш сохранён:', CACHE)

print('итог:', X_calib.shape, f'домен mean={X_calib.mean():.3f} std={X_calib.std():.3f}')
print('RAM:', ram())


In [ ]:

# ---------- Ячейка 6: модель -> ONNX -> OpenVINO IR FP32 (или приём готовой) ----------
if stage_done('export') and os.path.isfile('student_fp32.xml'):
    print('этап export уже сделан — пропускаю')
elif MODEL_INPUT == 'checkpoint':
    if ARCH_OVERRIDE:
        ak = plans['configurations'][CONFIG_NAME]['architecture']['arch_kwargs']
        for k, v in ARCH_OVERRIDE.items():
            ak[k] = v
        print('ARCH_OVERRIDE применён:', ARCH_OVERRIDE)
    json.dump(plans, open(os.path.join(WORK, 'plans_used.json'), 'w'), indent=1)

    # dataset.json: настоящий рядом (в маунтах) или универсальная реконструкция:
    # число выходных каналов читается из формы seg-головы чекпоинта (любой датасет)
    dj = find_any('/kaggle/input/**/dataset.json')
    if dj:
        dataset_json = json.load(open(dj))
        print('dataset.json:', dj)
    else:
        n_out = [v.shape[0] for k, v in state.items()
                 if 'seg_layers' in k and k.endswith('.weight')][0]
        labels = {'background': 0}
        labels.update({str(i + 1): i + 1 for i in range(n_out - 1)})
        dataset_json = {'channel_names': {'0': 'ct'}, 'labels': labels,
                        'numTraining': 1, 'file_ending': '.nii.gz'}
        print(f'dataset.json не найден — реконструкция: {n_out} выходных каналов')

    from nnunetv2.utilities.plans_handling.plans_handler import PlansManager
    from nnunetv2.training.nnUNetTrainer.nnUNetTrainer import nnUNetTrainer
    from nnunetv2.utilities.label_handling.label_handling import determine_num_input_channels

    pm = PlansManager(plans)
    cm = pm.get_configuration(CONFIG_NAME)
    lm = pm.get_label_manager(dataset_json)
    num_in = int(determine_num_input_channels(pm, cm, dataset_json))
    print('входных каналов:', num_in, '| сегментационных голов:', lm.num_segmentation_heads)
    # deep_supervision=False -> один выход полного разрешения (веса те же)
    net = nnUNetTrainer.build_network_architecture(
        cm.network_arch_class_name, cm.network_arch_init_kwargs,
        cm.network_arch_init_kwargs_req_import, num_in,
        lm.num_segmentation_heads, False)
    missing, unexpected = net.load_state_dict(state, strict=False)
    print(f'load_state_dict: missing={len(missing)}, unexpected={len(unexpected)}')
    assert not missing, f'не хватило весов: {missing[:5]}'
    net.eval()

    # opset 18 (dynamo-экспортёр torch 2.11; opset 11 не достижим — version
    # converter падает на Shape из opset 15). OV 2025.2/2026.4 понимают opset 18.
    dummy = torch.randn(1, num_in, *PATCH)
    t0 = time.time()
    torch.onnx.export(net, dummy, 'student.onnx', export_params=True,
                      opset_version=18, do_constant_folding=True,
                      input_names=['input'], output_names=['output'])
    print(f'ONNX: {time.time() - t0:.0f} с')
    # dynamo-экспортёр torch>=2.9 уводит веса в external data — переэкспорт single-file
    if os.path.isfile('student.onnx.data'):
        print('!! веса в external data — переэкспортирую single-file')
        os.remove('student.onnx')
        os.remove('student.onnx.data')
        ok = False
        try:  # dynamo-экспортёр (параметра save_as_external_data не существует)
            torch.onnx.export(net, dummy, 'student.onnx', export_params=True,
                              opset_version=18, do_constant_folding=True,
                              input_names=['input'], output_names=['output'],
                              external_data=False, dynamo=True)
            ok = True
        except TypeError:
            pass
        if not ok:  # legacy-экспортёр
            torch.onnx.export(net, dummy, 'student.onnx', export_params=True,
                              opset_version=18, do_constant_folding=True,
                              input_names=['input'], output_names=['output'],
                              save_as_external_data=False)
        assert not os.path.isfile('student.onnx.data'), 'веса всё ещё external'

    ov_model = ov.convert_model('student.onnx')
    ov.serialize(ov_model, 'student_fp32.xml')
    print('IR FP32 готов; выходов:', len(ov_model.outputs))
    mark_stage('export')
else:
    if MODEL_INPUT == 'onnx':
        ov_model = ov.convert_model(MODEL_FILE)
        ov.serialize(ov_model, 'student_fp32.xml')
    else:  # готовый IR: xml + bin рядом
        shutil.copy(MODEL_FILE, 'student_fp32.xml')
        bin_src = os.path.splitext(MODEL_FILE)[0] + '.bin'
        assert os.path.isfile(bin_src), f'нет {bin_src} рядом с xml'
        shutil.copy(bin_src, 'student_fp32.bin')
    print('IR FP32 готов из', MODEL_FILE)
    mark_stage('export')
print('RAM:', ram())


In [ ]:

# ---------- Ячейка 7: NNCF INT8 (subprocess — nbclient-таймауты не действуют) ----------
# Kaggle-nbclient рвёт молчащую ячейку (CellTimeoutError): NNCF выполняется
# отдельным процессом runner_quant.py, stdout стримится построчно — прогресс
# живой, таймауты не страшны. 256 патчей x ~12 с/патч (4 vCPU) ~= 50 мин.
out_xml = 'quantized_model.xml'
if stage_done('quant') and os.path.isfile(out_xml):
    K = min(K_CALIB, len(X_calib))
    print('этап quant уже сделан — пропускаю')
else:
    runner_code = chr(10).join([
        'import functools, json, os, sys, time',
        'print = functools.partial(print, flush=True)',
        'import numpy as np',
        "os.environ.setdefault('TF_CPP_MIN_LOG_LEVEL', '3')",
        "sys.modules.setdefault('tensorflow', None)",
        "sys.modules.setdefault('tensorboard', None)",
        'import nncf',
        'import openvino as ov',
        "print(f'[runner] стек: nncf {nncf.__version__} | openvino {ov.__version__}')",
        'K = int(sys.argv[1])',
        'out_xml = sys.argv[2]',
        "_cal = np.load('calibration_patches.npz')['patches'].astype(np.float32)",
        'K = min(K, len(_cal))',
        "print(f'runner: NNCF MIXED на {K} патчах {_cal.shape[1:]}')",
        'core = ov.Core()',
        "model = core.read_model('student_fp32.xml')",
        'try:',
        "    _ish = [d.get_length() if hasattr(d, 'get_length') else int(d) for d in model.input(0).shape]",
        "    assert all(v > 0 for v in _ish), f'динамический вход: {_ish}'",
        "    assert _ish == [1, 1, *_cal.shape[1:]], f'форма патча {_cal.shape[1:]} != вход модели {_ish}'",
        "    print('runner: форма входа модели', _ish)",
        'except Exception as e:',
        "    print('runner: предупреждение — сверка формы входа пропущена:', e)",
        'def gen():',
        '    for i in range(K):',
        '        print(f"[NNCF] патч {i+1}/{K}", flush=True)',
        '        yield _cal[i][np.newaxis]',
        't0 = time.time()',
        'q = nncf.quantize(model, nncf.Dataset(gen()),',
        '                  preset=nncf.QuantizationPreset.MIXED, subset_size=K)',
        'print(f"runner: NNCF {time.time()-t0:.0f} с")',
        'ov.serialize(q, out_xml)',
        "print('runner: сериализовано', out_xml)",
    ])
    with open('runner_quant.py', 'w') as f:
        f.write(runner_code)

    K = min(K_CALIB, len(X_calib))
    print(f'запуск runner: K={K} -> {out_xml} (ожидание ~{K * 12 // 60} мин на 4 vCPU)')
    proc = subprocess.Popen([sys.executable, 'runner_quant.py', str(K), out_xml],
                            stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                            text=True, bufsize=1)
    for ln in proc.stdout:
        print(ln.rstrip())
    proc.wait()
    assert proc.returncode == 0 and os.path.isfile(out_xml), (
        f'runner упал (код {proc.returncode}) — см. вывод выше')
    mark_stage('quant')
    print('RAM после runner:', ram())


In [ ]:

# ---------- Ячейка 8: гейт argmax INT8 vs FP32 на held-out патчах ----------
# Гейт меряет «INT8 сохранила FP32», НЕ качество модели (потерю квантизации
# мерять сравнением FP32 vs INT8 одной модели на валидации). Патчи гейта не
# участвовали в NNCF — честная проверка покрытия калибровки. Если гейт-партия
# пуста (редкий край), fallback на первые калибровочные патчи.
core = ov.Core()
fp32 = core.compile_model(core.read_model('student_fp32.xml'), 'CPU')
int8 = core.compile_model(core.read_model(out_xml), 'CPU')
n_gate = min(N_GATE, len(X_gate)) or min(N_GATE, len(X_calib))
agree_all = []
for i in range(n_gate):
    p = (X_gate[i] if i < len(X_gate) else X_calib[i])[np.newaxis]
    a = fp32(p)[fp32.output(0)][0].argmax(0)
    b = int8(p)[int8.output(0)][0].argmax(0)
    agree_all.append(float((a == b).mean()))
    print(f'патч {i}: совпадение argmax {agree_all[-1]:.3f}')
    del a, b, p
mean_agree = float(np.mean(agree_all))
print(f'СРЕДНЕЕ совпадение argmax ({n_gate} held-out патчей): {mean_agree:.4f}')
if mean_agree < 0.9:
    print('!! ГЕЙТ НЕ ПРОЙДЕН — квантизация сомнительна; проверьте домен/покрытие калибровки')
else:
    print('ГЕЙТ ПРОЙДЕН')


In [ ]:

# ---------- Ячейка 9: скорость + метаданные + итог ----------
inp = np.zeros((1, 1) + PATCH, np.float32)
times = {}
for c_, tag in ((fp32, 'FP32'), (int8, f'INT8({K}п)')):
    for _ in range(2):
        c_(inp)
    t0 = time.perf_counter()
    for _ in range(5):
        c_(inp)
    times[tag] = (time.perf_counter() - t0) / 5
    print(f'{tag}: {times[tag]:.2f} с/патч (Kaggle CPU 4 vCPU)')
t32, t8 = times['FP32'], times[f'INT8({K}п)']
print(f'ускорение: {t32 / t8:.2f}x (внутри одного прогона; между прогонами Kaggle несравнимо)')

meta = {'model_input': MODEL_INPUT, 'checkpoint': CKPT,
        'trainer_name': trainer_name, 'epoch': str(epoch),
        'config': CONFIG_NAME, 'features_per_stage': features,
        'arch_override': ARCH_OVERRIDE, 'patch_size': list(PATCH),
        'stack': STACK, 'calib_requested': K_CALIB, 'calib_used': int(len(X_calib)),
        'gate_patches': int(len(X_gate)), 'argmax_agree': mean_agree,
        'sec_per_patch': times, 'speedup': t32 / t8, 'timestamp': STAMP}
with open('model_meta.json', 'w') as f:
    json.dump(meta, f, indent=1)

print()
print(f'===== ГОТОВО: {trainer_name} (эпоха {epoch}) | INT8 на {K} показательных патчах =====')
print('Скачайте из Output: quantized_model.xml, quantized_model.bin, student_fp32.xml,')
print('  model_meta.json, selection_report.json')
print('Укладка: models\\headandneck\\<имя>\\fold_0\\ (quantized_model.xml/bin рядом с checkpoint)')
print('RAM финал:', ram())


In [ ]:

# ---------- Ячейка: CBPD-отчёт — стал ли deconv исполняемым в int8? ----------
import openvino as ov
_core = ov.Core()
for tag, path in [('INT8', 'quantized_model.xml'), ('FP32', 'student_fp32.xml')]:
    try:
        _m = _core.read_model(path)
    except Exception as _e:
        print(tag, 'read fail:', _e)
        continue
    _cbpd = [n for n in _m.get_ops() if n.get_type_name() == 'ConvolutionBackpropData']
    _fq = [n for n in _m.get_ops() if n.get_type_name() == 'FakeQuantize']
    _conv = [n for n in _m.get_ops() if n.get_type_name() == 'Convolution']
    print(f'CBPD_REPORT [{tag}]: CBPD={len(_cbpd)} FQ={len(_fq)} Conv={len(_conv)}')
    for _n in _cbpd:
        _i0, _i1 = _n.input(0), _n.input(1)
        print('  CBPD in0', str(_i0.get_element_type()), '<-', _i0.get_source_output().get_node().get_type_name(),
              '| w', str(_i1.get_element_type()), '<-', _i1.get_source_output().get_node().get_type_name())
print('CBPD_REPORT_END')
